# Пользователи, когорты и retention

Ноутбук готовит данные для дашборда retention и проверяет, как доля
рекомендаций в первый день связана с возвращаемостью пользователей.

## Методика

Когорта определяется по дню первого прослушивания. D1, D7 и D30 -
наличие хотя бы одного прослушивания ровно через 1, 7 или 30 дней.
Доля рекомендаций фиксируется только в первый день, чтобы не использовать
будущее поведение при формировании сегмента.

Дополнительно учитывается начальная активность: 1–10, 11–30, 31–60
или 61+ прослушиваний в первый день. Анализ показывает связь, а не
причинное влияние рекомендаций.

In [1]:
from pathlib import Path
import sys

import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

STAGE_DB = PROJECT_ROOT / "data" / "interim" / "yambda_stage.duckdb"
MARTS = PROJECT_ROOT / "data" / "processed"
assert STAGE_DB.exists(), "Сначала выполните ноутбук 01_source_quality.ipynb"
MARTS.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
con = duckdb.connect()

## Сборка витрин

In [2]:
from src.user_mart import build_retention_mart, build_user_day_mart, build_user_mart

USER_MART = MARTS / "mart_user.parquet"
USER_DAY_MART = MARTS / "mart_user_day.parquet"
RETENTION_MART = MARTS / "mart_retention_cohort.parquet"

user_result = build_user_mart(STAGE_DB, USER_MART)
user_day_result = build_user_day_mart(STAGE_DB, USER_DAY_MART)
retention_result = build_retention_mart(STAGE_DB, RETENTION_MART)
{
    "пользователи": user_result,
    "пользовательские дни": user_day_result,
    "retention": retention_result,
}

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{'пользователи': {'rows': 10000, 'sessions': 2529801},
 'пользовательские дни': {'rows': 1075989, 'users': 10000},
 'retention': {'rows': 487431}}

## Размер и активность аудитории

In [3]:
user_summary = con.execute(f'''
SELECT
    count(*) AS users,
    median(active_days) AS median_active_days,
    avg(active_days) AS avg_active_days,
    median(listens) AS median_listens,
    median(sessions) AS median_sessions,
    avg(recommendation_share) * 100 AS avg_user_recommendation_pct
FROM read_parquet('{USER_MART.as_posix()}')
''').df().round(2)
user_summary.rename(columns={
    "users": "пользователи", "median_active_days": "медиана активных дней",
    "avg_active_days": "среднее активных дней",
    "median_listens": "медиана прослушиваний",
    "median_sessions": "медиана сессий",
    "avg_user_recommendation_pct": "средняя доля рекомендаций, %",
})

,пользователи,медиана активных дней,среднее активных дней,медиана прослушиваний,медиана сессий,"средняя доля рекомендаций, %"
0,10000,86.00,107.60,"2,571.50",156.00,45.95


## Общий D1, D7 и D30

In [4]:
overall_retention = con.execute(f'''
SELECT
    lifetime_day,
    sum(retained_users) AS retained_users,
    sum(cohort_users) AS eligible_users,
    sum(retained_users) * 100.0 / sum(cohort_users) AS retention_pct
FROM read_parquet('{RETENTION_MART.as_posix()}')
WHERE lifetime_day IN (1, 7, 30)
GROUP BY lifetime_day
ORDER BY lifetime_day
''').df().round(2)
overall_retention.rename(columns={
    "lifetime_day": "день жизни", "retained_users": "вернулись",
    "eligible_users": "доступно для расчёта", "retention_pct": "retention, %",
})

,день жизни,вернулись,доступно для расчёта,"retention, %"
0,1,"5,052.00","9,238.00",54.69
1,7,"4,593.00","9,238.00",49.72
2,30,"3,987.00","8,944.00",44.58


## Retention по доле рекомендаций первого дня

In [5]:
retention_by_recommendations = con.execute(f'''
SELECT
    recommendation_bucket,
    sum(cohort_users) FILTER (WHERE lifetime_day = 0) AS users,
    sum(avg_first_day_listens * cohort_users)
        FILTER (WHERE lifetime_day = 0)
        / sum(cohort_users) FILTER (WHERE lifetime_day = 0)
        AS avg_first_day_listens,
    sum(retained_users) FILTER (WHERE lifetime_day = 1) * 100.0
        / sum(cohort_users) FILTER (WHERE lifetime_day = 1) AS d1_pct,
    sum(retained_users) FILTER (WHERE lifetime_day = 7) * 100.0
        / sum(cohort_users) FILTER (WHERE lifetime_day = 7) AS d7_pct,
    sum(retained_users) FILTER (WHERE lifetime_day = 30) * 100.0
        / sum(cohort_users) FILTER (WHERE lifetime_day = 30) AS d30_pct
FROM read_parquet('{RETENTION_MART.as_posix()}')
GROUP BY recommendation_bucket
ORDER BY CASE recommendation_bucket
    WHEN '0%' THEN 1 WHEN '1-25%' THEN 2 WHEN '25-50%' THEN 3
    WHEN '50-75%' THEN 4 WHEN '75-99%' THEN 5 ELSE 6 END
''').df().round(2)
retention_by_recommendations.rename(columns={
    "recommendation_bucket": "доля рекомендаций", "users": "пользователи",
    "avg_first_day_listens": "прослушивания в первый день",
    "d1_pct": "D1, %", "d7_pct": "D7, %", "d30_pct": "D30, %",
})

,доля рекомендаций,пользователи,прослушивания в первый день,"D1, %","D7, %","D30, %"
0,0%,"3,511.00",25.78,43.29,41.27,36.72
1,1-25%,"1,015.00",68.57,67.98,58.33,53.88
2,25-50%,810.00,60.48,72.47,61.11,54.94
3,50-75%,658.00,65.88,74.32,63.07,55.50
4,75-99%,760.00,75.33,68.29,62.63,57.63
5,100%,"2,484.00",30.79,50.20,46.94,42.01


## D7 с учётом начальной активности

In [6]:
controlled_d7 = con.execute(f'''
SELECT
    engagement_bucket,
    recommendation_bucket,
    sum(cohort_users) AS users,
    sum(retained_users) * 100.0 / sum(cohort_users) AS d7_pct
FROM read_parquet('{RETENTION_MART.as_posix()}')
WHERE lifetime_day = 7
GROUP BY engagement_bucket, recommendation_bucket
ORDER BY CASE engagement_bucket
           WHEN '1-10' THEN 1 WHEN '11-30' THEN 2
           WHEN '31-60' THEN 3 ELSE 4 END,
         CASE recommendation_bucket
           WHEN '0%' THEN 1 WHEN '1-25%' THEN 2 WHEN '25-50%' THEN 3
           WHEN '50-75%' THEN 4 WHEN '75-99%' THEN 5 ELSE 6 END
''').df()

controlled_d7.pivot(
    index="engagement_bucket",
    columns="recommendation_bucket",
    values="d7_pct",
).round(2).rename_axis("прослушивания в первый день")

recommendation_bucket,0%,1-25%,100%,25-50%,50-75%,75-99%
прослушивания в первый день,,,,,,
1-10,29.43,42.55,36.78,43.26,53.03,52.17
11-30,44.79,50.68,46.38,62.38,53.42,55.68
31-60,60.47,60.53,55.99,59.39,62.28,54.82
61+,58.59,66.94,62.97,70.99,71.97,72.14


## Когорты для тепловой карты

In [7]:
cohort_preview = con.execute(f'''
SELECT
    cohort_day,
    sum(cohort_users) FILTER (WHERE lifetime_day = 0) AS cohort_users,
    sum(retained_users) FILTER (WHERE lifetime_day = 1) * 100.0
        / sum(cohort_users) FILTER (WHERE lifetime_day = 1) AS d1_pct,
    sum(retained_users) FILTER (WHERE lifetime_day = 7) * 100.0
        / sum(cohort_users) FILTER (WHERE lifetime_day = 7) AS d7_pct,
    sum(retained_users) FILTER (WHERE lifetime_day = 30) * 100.0
        / sum(cohort_users) FILTER (WHERE lifetime_day = 30) AS d30_pct
FROM read_parquet('{RETENTION_MART.as_posix()}')
GROUP BY cohort_day
HAVING d30_pct IS NOT NULL
ORDER BY cohort_day
LIMIT 10
''').df().round(2)
cohort_preview.rename(columns={
    "cohort_day": "когортный день", "cohort_users": "размер когорты",
    "d1_pct": "D1, %", "d7_pct": "D7, %", "d30_pct": "D30, %",
})

,когортный день,размер когорты,"D1, %","D7, %","D30, %"
0,0,"2,542.00",66.72,72.07,61.01
1,1,640.00,48.28,46.41,48.75
2,2,313.00,41.85,39.62,40.26
3,3,262.00,44.66,37.79,39.31
4,4,175.00,43.43,36.00,32.57
5,5,113.00,43.36,35.40,30.09
6,6,102.00,37.25,31.37,28.43
7,7,113.00,30.97,28.32,27.43
8,8,89.00,28.09,29.21,29.21
9,9,56.00,26.79,26.79,30.36


## Проверки

In [8]:
user_day_check = con.execute(f'''
SELECT
    count(*) = count(DISTINCT (uid, day_idx)) AS unique_key,
    sum(listens) = 46467212 AS listens_match
FROM read_parquet('{USER_DAY_MART.as_posix()}')
''').df()
retention_check = con.execute(f'''
SELECT
    count(*) = count(DISTINCT (
        cohort_day, lifetime_day, recommendation_bucket, engagement_bucket
    )) AS unique_key,
    min(retention_rate) >= 0 AND max(retention_rate) <= 1 AS rates_valid,
    count(*) FILTER (WHERE lifetime_day = 0 AND retention_rate <> 1) = 0
        AS d0_valid
FROM read_parquet('{RETENTION_MART.as_posix()}')
''').df()
assert user_day_check.all(axis=None) and retention_check.all(axis=None)
print("Проверки пройдены: пользовательские дни и retention рассчитаны корректно.")

Проверки пройдены: пользовательские дни и retention рассчитаны корректно.


## Что готово для дашборда

- карточки D1, D7 и D30;
- когортная тепловая карта;
- retention-кривые по дням жизни;
- сравнение сегментов рекомендаций;
- контроль стартовой активности пользователя.

Различия между сегментами остаются наблюдательной связью. Для вывода о
причинном влиянии рекомендаций понадобится эксперимент или более строгий
квазиэкспериментальный дизайн.

In [9]:
con.close()
print("Соединение закрыто.")

Соединение закрыто.
